# Exploring Hackers News Posts
In this project, I will compare two different types of posts from Hacker News, a popular site for voting and commenting on stories (or “posts”) related to technology. The two types of posts I will examine begin with Ask HN or Show HN.

Users submit `Ask HN` posts to ask the Hacker News community a specific question, such as “What is the best online course you have ever taken?” Similarly, users submit `Show HN` posts to show the Hacker News community a project, a product, or just something interesting in general.

I will specifically compare these two types of posts to determine the following:
* Do Ask HN or Show HN get more comments on average?
* Do posts created at certain times of the day get more comments on average?

It should be noted that the dataset I will be working with was reduced from almost 300,000 rows to about 20,000 rows by removing all posts that did not receive any comments and then randomly selecting from the remaining posts.

## Read in the data

In [1]:
from csv import reader

with open('hacker_news.csv') as opened_file:
    hn_data = list(reader(opened_file))

# View the first five rows (including headers)
hn_data[:5]

[['id', 'title', 'url', 'num_points', 'num_comments', 'author', 'created_at'],
 ['12224879',
  'Interactive Dynamic Video',
  'http://www.interactivedynamicvideo.com/',
  '386',
  '52',
  'ne0phyte',
  '8/4/2016 11:52'],
 ['10975351',
  'How to Use Open Source and Shut the Fuck Up at the Same Time',
  'http://hueniverse.com/2016/01/26/how-to-use-open-source-and-shut-the-fuck-up-at-the-same-time/',
  '39',
  '10',
  'josep2',
  '1/26/2016 19:30'],
 ['11964716',
  "Florida DJs May Face Felony for April Fools' Water Joke",
  'http://www.thewire.com/entertainment/2013/04/florida-djs-april-fools-water-joke/63798/',
  '2',
  '1',
  'vezycash',
  '6/23/2016 22:20'],
 ['11919867',
  'Technology ventures: From Idea to Enterprise',
  'https://www.amazon.com/Technology-Ventures-Enterprise-Thomas-Byers/dp/0073523429',
  '3',
  '1',
  'hswarna',
  '6/17/2016 0:01']]

## Removing Headers from a List of Lists

In [2]:
hn_headers = hn_data[0]
hn_data = hn_data[1:]
print(hn_headers)
hn_data[:5]

['id', 'title', 'url', 'num_points', 'num_comments', 'author', 'created_at']


[['12224879',
  'Interactive Dynamic Video',
  'http://www.interactivedynamicvideo.com/',
  '386',
  '52',
  'ne0phyte',
  '8/4/2016 11:52'],
 ['10975351',
  'How to Use Open Source and Shut the Fuck Up at the Same Time',
  'http://hueniverse.com/2016/01/26/how-to-use-open-source-and-shut-the-fuck-up-at-the-same-time/',
  '39',
  '10',
  'josep2',
  '1/26/2016 19:30'],
 ['11964716',
  "Florida DJs May Face Felony for April Fools' Water Joke",
  'http://www.thewire.com/entertainment/2013/04/florida-djs-april-fools-water-joke/63798/',
  '2',
  '1',
  'vezycash',
  '6/23/2016 22:20'],
 ['11919867',
  'Technology ventures: From Idea to Enterprise',
  'https://www.amazon.com/Technology-Ventures-Enterprise-Thomas-Byers/dp/0073523429',
  '3',
  '1',
  'hswarna',
  '6/17/2016 0:01'],
 ['10301696',
  'Note by Note: The Making of Steinway L1037 (2007)',
  'http://www.nytimes.com/2007/11/07/movies/07stein.html?_r=0',
  '8',
  '2',
  'walterbell',
  '9/30/2015 4:12']]

We can see above that the dataset contains the post `titles`, the `number of comments` on each post, and the `date` the post was created. Let's start by examining the number of comments on each type of post.

##  Extracting Ask HN and Show HN Posts
First, I will identify posts whose name begins with `Ask HN` or `Show HN` and separate the data for these two types of posts into different lists. Separating the data will make it easier to analyze in the next steps.

In [3]:
ask_posts = []
show_posts = []
other_posts = []

for post in hn_data:
    title = post[1]
    # For clarity, I will lowercase the titles
    if title.lower().startswith('ask hn'):
        ask_posts.append(post)
    elif title.lower().startswith('show hn'):
        show_posts.append(post)
    else:
        other_posts.append(post)

# I will count the number of posts for each category
print(f"Number of posts for 'Ask HN' category: {len(ask_posts)}")
print(f"Number of posts for 'Show HN' category: {len(show_posts)}")
print(f"Number of posts for 'Other' category: {len(other_posts)}")

Number of posts for 'Ask HN' category: 1744
Number of posts for 'Show HN' category: 1162
Number of posts for 'Other' category: 17194


## Calculating the Average Number of Comments for Ask HN and Show HN Posts
Now that we separated ask posts and show posts into different lists, we'll calculate the average number of comments each type of post receives.

In [4]:
# Calculate the average number of comments `Ask HN` posts receive
total_ask_comments = 0
for post in ask_posts:
    n_comments = int(post[4])
    total_ask_comments += n_comments

avg_ask_comments = total_ask_comments / len(ask_posts)
print(f'Average number of comments for Ask HN: {avg_ask_comments}')

Average number of comments for Ask HN: 14.038417431192661


In [5]:
# Calculate the average number of comments `Show HN` posts receive
total_show_comments = 0
for post in show_posts:
    n_comments = int(post[4])
    total_show_comments += n_comments

avg_show_comments = total_show_comments / len(show_posts)
print(f'Average number of comments for Show HN: {avg_show_comments}')

Average number of comments for Show HN: 10.31669535283993


On average, ask posts in sample receive approximately 14 comments, whereas show posts receive approximately 10. Since ask posts are more likely to receive comments, I'll focus remaining analysis just on these posts.

## Finding the Amount of Ask Posts and Comments by Hour Created
Next, I'll determine if I can maximize the amount of comments an ask post receives by creating it at a certain time. First, I'll find the amount of ask posts created during each hour of day, along with the number of comments those posts received. Then, I'll calculate the average amount of comments ask posts created at each hour of the day receive.

In [6]:
import datetime as dt

# I will choose the hours of creating posts and the number of comments on these posts, save them in a separate list
result_list = []

for row in ask_posts:
    result_list.append([row[-1], int(row[4])])

posts_by_hour = {}
comments_by_hour = {}
date_format = '%m/%d/%Y %H:%M'  # I will set the date format for parsing a date from a string.

for row in result_list:
    time = dt.datetime.strptime(row[0], date_format).strftime('%H')
    comments = int(row[1])
    if time not in posts_by_hour:
        posts_by_hour[time] = 1
        comments_by_hour[time] = comments
    else:
        posts_by_hour[time] += 1
        comments_by_hour[time] += comments

## Calculating the Average Number of Comments for Ask HN Posts by Hour

In [7]:
# Calculate the average amount of comments `Ask HN` posts created at each hour of the day receive
avg_by_hour = []
for hour in comments_by_hour:
    avg_by_hour.append([hour, comments_by_hour[hour] / posts_by_hour[hour]])

avg_by_hour

[['09', 5.5777777777777775],
 ['13', 14.741176470588234],
 ['10', 13.440677966101696],
 ['14', 13.233644859813085],
 ['16', 16.796296296296298],
 ['23', 7.985294117647059],
 ['12', 9.41095890410959],
 ['17', 11.46],
 ['15', 38.5948275862069],
 ['21', 16.009174311926607],
 ['20', 21.525],
 ['02', 23.810344827586206],
 ['18', 13.20183486238532],
 ['03', 7.796296296296297],
 ['05', 10.08695652173913],
 ['19', 10.8],
 ['01', 11.383333333333333],
 ['22', 6.746478873239437],
 ['08', 10.25],
 ['04', 7.170212765957447],
 ['00', 8.127272727272727],
 ['06', 9.022727272727273],
 ['07', 7.852941176470588],
 ['11', 11.051724137931034]]

## Sorting and Printing Values from a List of Lists

In [8]:
swap_avg_by_hour = []
for row in avg_by_hour:
    swap_avg_by_hour.append([row[1], row[0]])

sorted_swap = sorted(swap_avg_by_hour, reverse=True)
sorted_swap

[[38.5948275862069, '15'],
 [23.810344827586206, '02'],
 [21.525, '20'],
 [16.796296296296298, '16'],
 [16.009174311926607, '21'],
 [14.741176470588234, '13'],
 [13.440677966101696, '10'],
 [13.233644859813085, '14'],
 [13.20183486238532, '18'],
 [11.46, '17'],
 [11.383333333333333, '01'],
 [11.051724137931034, '11'],
 [10.8, '19'],
 [10.25, '08'],
 [10.08695652173913, '05'],
 [9.41095890410959, '12'],
 [9.022727272727273, '06'],
 [8.127272727272727, '00'],
 [7.985294117647059, '23'],
 [7.852941176470588, '07'],
 [7.796296296296297, '03'],
 [7.170212765957447, '04'],
 [6.746478873239437, '22'],
 [5.5777777777777775, '09']]

In [9]:
# Sort the values and print the the 5 hours with the highest average comments
print("Top 5 Hours for 'Ask HN' Comments")
for row in sorted_swap[:5]:
    time = dt.datetime.strptime(row[1], '%H').strftime('%H:%M')
    avg_comments = row[0]
    print(f'{time}: {avg_comments:.2f} average comments per post')

Top 5 Hours for 'Ask HN' Comments
15:00: 38.59 average comments per post
02:00: 23.81 average comments per post
20:00: 21.52 average comments per post
16:00: 16.80 average comments per post
21:00: 16.01 average comments per post


The hour that receives the most comments per post on average is 15:00, with an average of 38.59 comments per post. There's about a 60% increase in the number of comments between the hours with the highest and second highest 
average number of comments.

According to the data set [documentation](https://www.kaggle.com/hacker-news/hacker-news-posts/home), the timezone used is Eastern Time in the US. So, we could also write 15:00 as 3:00 pm est.

## Conclusion
In this project, I analyzed Ask HN and Show HN posts to determine which post type and time of day, on average, received the most comments.

Ask HN posts received more comments than Show HN posts, on average, about 14 comments per Ask HN post compared to about 10 comments per Show HN post.

Among Ask HN posts, those created at 3:00 PM EST received the highest average number of comments, at 38.59 comments per post.

However, these results should be interpreted with caution, as the dataset excludes posts that received no comments. Therefore, the analysis describes the average number of comments among posts that received at least one comment, not the probability that a newly created post will receive comments.